# *Nonlinear Arterial Hemodynamics*
## Chapter 5 companion — Geometry-Parameterized Spectral Dynamics

This notebook is the computational companion to Chapter 5. It reproduces the canonical reduced axial model and uses VascuQuest/PWDB only to place its prescribed geometry-sensitive coefficients and Womersley dependence in physiological context.

The book nomenclature governs all reader-facing quantities.

The notebook restores scalar Newtonian rheology and a rigid wall. It admits weak axial evolution, quadratic steepening, dispersion, nonlocal damping, and redistribution among axial Fourier wavenumbers. Constitutive anisotropy is withheld, and wall compliance is not introduced until Chapter 6.

A strict distinction is maintained between:

- resolved arterial geometry supplied by VascuQuest;
- the prescribed geometry-to-coefficient interface $\mathcal M_G$;
- the spatially averaged coefficients actually advanced by the canonical solver.

**Execution:** a clean Google Colab runtime should reproduce all outputs using **Run all** with no manual choices.

### Chapter question

Chapter 5 asks whether a weak axial perturbation can redistribute quadratic energy toward shorter axial scales while the total perturbation energy still decays.

The notebook therefore has five responsibilities:

1. reconstruct the reduced axial amplitude model and its Fourier mechanics;
2. reproduce the canonical Case B verification benchmark;
3. separate spectral redistribution from global energy growth or decay;
4. test mechanism-off counterfactuals and parameter dependence;
5. use VascuQuest to show the range of arterial geometry and Womersley conditions to which a **separately prescribed** reduced coefficient map might eventually be connected.

The notebook does not claim that PWDB geometry directly determines $b$ or $g$.

### VascuQuest representation

VascuQuest/PWDB is used in two limited ways.

First, source geometry for a deterministic representative virtual subject supplies segment-level quantities such as length and inlet/outlet radius. These are used only to show that arterial geometry spans multiple axial and radial scales. No universal map from those descriptors to $b(\zeta)$ or $g(\zeta)$ is inferred.

Second, PWDB heart rate and time-mean luminal area provide

$$
R=\sqrt{\frac{\langle A\rangle_t}{\pi}},
\qquad
\Omega=\frac{2\pi}{T},
$$

and therefore

$$
\alpha=R\sqrt{\frac{\Omega}{\nu}}.
$$

The Chapter 5 canonical parameterization then gives

$$
b_0=b_{\mathrm{ref}}\alpha^{-2},
$$

$$
g_0=g_{\mathrm{ref}}
\left(1+\frac{C_g}{\alpha}\right),
$$

with the canonical values

$$
b_{\mathrm{ref}}=1,
\qquad
g_{\mathrm{ref}}=0.005,
\qquad
C_g=0.1.
$$

This is a **book-defined reduced-model projection** of VascuQuest-derived $\alpha$, not a calibration of arterial geometry.

In [ ]:
# Configuration and reproducibility constants
from pathlib import Path
import sys, json, subprocess, zipfile

ROOT = Path("/content/nonlinear_arterial_hemodynamics_ch05")
FIG_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"
META_DIR = ROOT / "metadata"
for d in (ROOT, FIG_DIR, DATA_DIR, META_DIR):
    d.mkdir(parents=True, exist_ok=True)

VQ_REPOSITORY = "https://github.com/KNOWDYN/VascuQuest.git"
VQ_GIT_REF = "8307147d72e7a6f3ea3135895bd6f52927c67439"
PWDB_RECORD_ID = "3275625"
PWDB_DOI = "10.5281/zenodo.3275625"

rho = 1060.0       # kg m^-3
mu = 3.5e-3        # Pa s
nu = mu / rho      # m^2 s^-1

# Canonical Chapter 5 reduced-model parameters.
L_g = 4.0 * 3.141592653589793
alpha_caseB = 10.0
b_ref = 1.0
g_ref = 0.005
C_g = 0.1
b_caseB = b_ref * alpha_caseB**-2
g_caseB = g_ref * (1.0 + C_g/alpha_caseB)
kappa_c = 2.0

SITES = [
    "AorticRoot", "ThorAorta", "AbdAorta",
    "Carotid", "Brachial", "Radial", "Femoral",
]

print("Working directory:", ROOT)
print(f"nu = {nu:.6e} m^2/s")
print(f"Case B: b_avg={b_caseB:.6f}, g_avg={g_caseB:.6f}")

In [ ]:
# Install the pinned VascuQuest revision.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    f"git+{VQ_REPOSITORY}@{VQ_GIT_REF}"
])

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator
import vascuquest as vq

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("VascuQuest:", getattr(vq, "__version__", "version field not exposed"))

In [ ]:
# Acquire and verify the exact PWDB artifacts used here.
ARTIFACTS = [
    "model_configurations",
    "geometry",
    "common_site_waveforms_csv",
]
verification = {}

for artifact in ARTIFACTS:
    subprocess.run(
        ["vascuquest", "dataset", "acquire",
         "--artifact", artifact, "--yes", "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verified = subprocess.run(
        ["vascuquest", "dataset", "verify",
         "--artifact", artifact, "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verification[artifact] = json.loads(verified.stdout)

status = subprocess.run(
    ["vascuquest", "dataset", "status", "--format", "json"],
    check=True, text=True, capture_output=True,
)
dataset_status = json.loads(status.stdout)
SOURCE_DIR = Path(dataset_status["managed_paths"]["source"])

(META_DIR / "artifact_verification.json").write_text(
    json.dumps(verification, indent=2), encoding="utf-8"
)
(META_DIR / "dataset_status.json").write_text(
    json.dumps(dataset_status, indent=2), encoding="utf-8"
)

print("Verified PWDB source:", SOURCE_DIR)

In [ ]:
# Open the verified dataset and establish deterministic subject metadata.
session = vq.open_dataset(source=SOURCE_DIR, offline=True)
assert session.identity.record_id == PWDB_RECORD_ID

age_result = session.get("age")
subject_ids = np.asarray(age_result.coordinates[0].values, dtype=str)
ages = np.asarray(age_result.values, dtype=float)

hr_result = session.get("heart_rate", subjects=subject_ids.tolist())
hr_ids = np.asarray(hr_result.coordinates[0].values, dtype=str)
heart_rates = np.asarray(hr_result.values, dtype=float)
assert np.array_equal(subject_ids, hr_ids)

subject_meta = pd.DataFrame({
    "subject_id": subject_ids,
    "age_years": ages,
    "heart_rate_bpm": heart_rates,
})
subject_meta["subject_number"] = subject_meta["subject_id"].astype(int)
subject_meta = subject_meta.sort_values("subject_number").reset_index(drop=True)

# Deterministic representative subject: middle source age stratum,
# then median canonical subject number within that stratum.
source_ages = sorted(subject_meta["age_years"].dropna().unique())
target_age = source_ages[len(source_ages)//2]
age_group = subject_meta.loc[subject_meta["age_years"] == target_age].copy()
age_group = age_group.sort_values("subject_number").reset_index(drop=True)
representative_subject = str(age_group.iloc[len(age_group)//2]["subject_id"])

selection_record = {
    "rule": "middle PWDB source age stratum; median canonical subject number",
    "representative_subject_id": representative_subject,
    "representative_age_years": float(target_age),
    "source_age_strata_years": [float(x) for x in source_ages],
}
(META_DIR / "representative_subject.json").write_text(
    json.dumps(selection_record, indent=2), encoding="utf-8"
)
display(pd.DataFrame([selection_record]))

In [ ]:
# Shared B&W plotting style and PWDB waveform reader.
WAVE_ZIP = SOURCE_DIR / "PWs_csv.zip"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9.0,
    "axes.labelsize": 9.0,
    "axes.titlesize": 9.5,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "legend.fontsize": 7.8,
    "axes.linewidth": 0.75,
    "lines.linewidth": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
BLACK, DARK, MID, LIGHT = "0.0", "0.28", "0.52", "0.74"

SITE_LABELS = {
    "AorticRoot": "Aortic root",
    "ThorAorta": "Thoracic aorta",
    "AbdAorta": "Abdominal aorta",
    "Carotid": "Carotid",
    "Brachial": "Brachial",
    "Radial": "Radial",
    "Femoral": "Femoral",
}

def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

def save_figure(fig, stem):
    pdf = FIG_DIR / f"{stem}.pdf"
    png = FIG_DIR / f"{stem}.png"
    fig.savefig(pdf, bbox_inches="tight", pad_inches=0.03)
    fig.savefig(png, dpi=600, bbox_inches="tight", pad_inches=0.03)
    return pdf, png

def _wave_member_name(site_id, source_signal):
    basename = f"PWs_{site_id}_{source_signal}.csv"
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        matches = [name for name in zf.namelist() if Path(name).name == basename]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {basename!r}; found {len(matches)}")
    return matches[0]

def load_waveform_matrix(site_id, source_signal):
    # Database-native field names are confined to this mapping layer.
    member = _wave_member_name(site_id, source_signal)
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        with zf.open(member, "r") as raw:
            frame = pd.read_csv(raw, low_memory=False)
    ids = np.asarray([str(int(x)) for x in frame.iloc[:, 0].to_numpy()], dtype=str)
    values = frame.iloc[:, 1:].to_numpy(dtype=float)
    return ids, values

print("Shared helpers ready.")

# Why a reduced axial model is introduced

The Womersley reference state is fully developed:

$$
\frac{\partial u_z}{\partial z}=0.
$$

Chapter 5 restores a weak axial degree of freedom through

$$
u_z(r,z,t)
=
u_W(r,t)
+
\epsilon a(z,t)\phi(r)
+
O(\epsilon^2),
\qquad
0<\epsilon\ll1.
$$

The radial projection uses

$$
\langle f,g\rangle_r
=
\int_0^R f(r)g(r)\,r\,dr,
\qquad
\langle\phi,\phi\rangle_r=1.
$$

Retained geometric information is represented abstractly by

$$
\{b(\zeta),g(\zeta)\}
=
\mathcal M_G[\mathcal G(z)].
$$

The notebook preserves the chapter's interpretation: $\mathcal M_G$ is a **modeling interface**, not a universal geometry law and not a homogenization theorem.

# VascuQuest geometry spans multiple arterial scales

The PWDB geometry artifact can document the scale of the geometry that a future calibrated $\mathcal M_G$ would have to represent.

For the deterministic representative subject, the notebook uses source segment:

- length;
- inlet radius;
- outlet radius.

From these, it computes only transparent descriptors:

$$
R_m=\frac{R_{\mathrm{in}}+R_{\mathrm{out}}}{2},
$$

$$
\frac{L}{R_m},
$$

and

$$
\frac{R_{\mathrm{in}}-R_{\mathrm{out}}}{R_m}.
$$

These descriptors are not converted into $b$ or $g$.

In [ ]:
# Source geometry for the deterministic representative subject.
geo_result = session.geometry(subject=representative_subject)

geo_rows = []
for seg in geo_result.values:
    R_m = 0.5*(seg.inlet_radius_m + seg.outlet_radius_m)
    taper = (seg.inlet_radius_m - seg.outlet_radius_m)/R_m
    slenderness = seg.length_m/R_m

    geo_rows.append({
        "segment_id": seg.segment_id,
        "length_m": seg.length_m,
        "R_in_m": seg.inlet_radius_m,
        "R_out_m": seg.outlet_radius_m,
        "R_m_m": R_m,
        "taper_ratio": taper,
        "slenderness_L_over_Rm": slenderness,
    })

geo_df = pd.DataFrame(geo_rows)
geo_df["length_mm"] = 1e3*geo_df["length_m"]
geo_df["R_m_mm"] = 1e3*geo_df["R_m_m"]
geo_df.to_csv(DATA_DIR / "ch05_representative_geometry.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.05))

axes[0].scatter(
    geo_df["R_m_mm"], geo_df["length_mm"],
    s=18, facecolors="none", edgecolors=BLACK, linewidths=0.7
)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel(r"Mean segment radius, $R_m$ (mm)")
axes[0].set_ylabel(r"Segment length, $L$ (mm)")
clean_axes(axes[0])

axes[1].scatter(
    geo_df["slenderness_L_over_Rm"], geo_df["taper_ratio"],
    s=18, facecolors="none", edgecolors=BLACK, linewidths=0.7
)
axes[1].set_xscale("log")
axes[1].axhline(0.0, color=LIGHT, linewidth=0.8)
axes[1].set_xlabel(r"Slenderness, $L/R_m$")
axes[1].set_ylabel(r"Taper, $(R_{\mathrm{in}}-R_{\mathrm{out}})/R_m$")
clean_axes(axes[1])

fig.tight_layout(w_pad=1.2)
save_figure(fig, "ch05_vascuquest_geometry_scales")
plt.show()

This is a VascuQuest geometry observation. It establishes only that the source arterial tree contains a broad range of segment scales and taper descriptors.

It does **not** establish how curvature, tortuosity, taper, or roughness should map into $b$ and $g$. That missing connection is exactly why the chapter keeps $\mathcal M_G$ explicit.

# The reduced equation and its mechanical terms

Introduce

$$
\zeta=\frac{z}{L_0},
\qquad
s=\frac{t}{T_0},
\qquad
a(z,t)=a_{\mathrm{ref}}\widetilde a(\zeta,s).
$$

The variable-coefficient model is

$$
\frac{\partial\widetilde a}{\partial s}
+
\widetilde a\frac{\partial\widetilde a}{\partial\zeta}
+
b(\zeta)\frac{\partial^3\widetilde a}{\partial\zeta^3}
+
g(\zeta)
\left(-\partial_\zeta^2\right)^{(1+c_0)/2}
\widetilde a
=
0.
$$

The canonical calculations set

$$
c_0=0,
$$

so the damping operator has Fourier symbol $|\kappa|$.

The terms have different jobs:

- $\widetilde a\,\widetilde a_\zeta$: nonlinear steepening and spectral convolution;
- $b\,\widetilde a_{\zeta\zeta\zeta}$: dispersion;
- $g(-\partial_\zeta^2)^{1/2}\widetilde a$: nonlocal damping.

Appendix I develops the nonlinear-wave and fractional-operator mechanics; Appendix G fixes the spatial Fourier and Parseval conventions.

# Geometry parameterization and implemented model

The conceptual variable coefficients are

$$
b(\zeta)
=
b_0
\left[
1+\epsilon_b\cos(\kappa_g\zeta)
\right],
$$

$$
g(\zeta)
=
g_0
\left[
1+\epsilon_g\cos(\kappa_g\zeta)
\right].
$$

Their baselines are parameterized by

$$
b_0=b_{\mathrm{ref}}\alpha^{-2},
$$

$$
g_0
=
g_{\mathrm{ref}}
\left(
1+\frac{C_g}{\alpha}
\right).
$$

However, the canonical numerical solver advances the **spatially averaged** system

$$
\widetilde a_s
+
\widetilde a\widetilde a_\zeta
+
b_{\mathrm{avg}}\widetilde a_{\zeta\zeta\zeta}
+
g_{\mathrm{avg}}
(-\partial_\zeta^2)^{1/2}\widetilde a
=
0,
$$

with

$$
b_{\mathrm{avg}}=b_0,
\qquad
g_{\mathrm{avg}}=g_0
$$

for the sinusoidal conceptual parameterization.

Therefore $\epsilon_b$, $\epsilon_g$, and $\kappa_g$ do not enter the canonical time stepping after spatial averaging. The notebook does not pretend otherwise.

# Spectral mechanics and diagnostics

For

$$
\widetilde a(\zeta,s)
=
\sum_\kappa
\widehat a_\kappa(s)e^{i\kappa\zeta},
$$

the linear operator is

$$
\mathcal L(\kappa)
=
+i b_{\mathrm{avg}}\kappa^3
-
g_{\mathrm{avg}}|\kappa|.
$$

The nonlinear term satisfies

$$
\mathcal F\{
\widetilde a\widetilde a_\zeta
\}_\kappa
=
\frac{i\kappa}{2}
\sum_{\kappa_1+\kappa_2=\kappa}
\widehat a_{\kappa_1}
\widehat a_{\kappa_2}.
$$

The quadratic energy is

$$
I_2(s)
=
\int_0^{L_g}
\widetilde a^2(\zeta,s)\,d\zeta.
$$

Under the chapter's Fourier normalization,

$$
I_2
=
L_g\sum_n|\widehat a_n|^2.
$$

The exact energy identity for the spatially averaged periodic model is

$$
\frac{dI_2}{ds}
=
-2g_{\mathrm{avg}}L_g
\sum_n
|\kappa_n|
|\widehat a_n|^2
\le0.
$$

The global logarithmic diagnostic is

$$
\Gamma_E(s)
=
\frac{d}{ds}\ln I_2(s).
$$

To diagnose redistribution, define

$$
E_{\mathrm{low}}
=
L_g
\sum_{0<|\kappa_n|\le\kappa_c}
|\widehat a_n|^2,
$$

$$
E_{\mathrm{high}}
=
L_g
\sum_{|\kappa_n|>\kappa_c}
|\widehat a_n|^2,
$$

and

$$
\mathcal R_{\mathrm{spec}}
=
\frac{E_{\mathrm{high}}}{E_{\mathrm{low}}}.
$$

For Case B the cutoff is $\kappa_c=2$, above the initially occupied band.

In [ ]:
# Split Fourier solver for the spatially averaged Chapter 5 equation.
#
# The linear dispersive/damping part is advanced exactly over half steps.
# The quadratic nonlinear term is advanced with RK4 in Fourier space.
# A 2/3 de-aliasing mask is applied to nonlinear products, following
# Appendix G and the Chapter 8 Case B verification.
def chapter5_grid(N):
    zeta = np.arange(N, dtype=float) * L_g/N
    kappa = 2.0*np.pi*np.fft.fftfreq(N, d=L_g/N)
    mode_number = np.fft.fftfreq(N)*N
    return zeta, kappa, mode_number

def initial_caseB(zeta):
    return (
        np.sin(0.5*zeta)
        + 0.3*np.sin(zeta)
        + 0.1*np.sin(1.5*zeta)
    )

def diagnostics_from_hat(a_hat, kappa):
    # numpy FFT normalization -> book Fourier-series coefficient a_hat/N.
    N = len(a_hat)
    coeff = a_hat/N

    I2 = L_g*np.sum(np.abs(coeff)**2)

    low = (np.abs(kappa) > 1e-14) & (np.abs(kappa) <= kappa_c + 1e-14)
    high = np.abs(kappa) > kappa_c + 1e-14

    E_low = L_g*np.sum(np.abs(coeff[low])**2)
    E_high = L_g*np.sum(np.abs(coeff[high])**2)
    R_spec = E_high/E_low if E_low > 0 else np.nan

    dissipation_sum = L_g*np.sum(np.abs(kappa)*np.abs(coeff)**2)
    return I2, E_low, E_high, R_spec, dissipation_sum

def run_ch5(
    N=512,
    dt=1e-3,
    s_end=10.0,
    b_avg=b_caseB,
    g_avg=g_caseB,
    nonlinear=True,
    dispersion=True,
    damping=True,
    dealias=True,
    save_every=50,
):
    zeta, kappa, mode_number = chapter5_grid(N)
    a0 = initial_caseB(zeta)

    b_use = b_avg if dispersion else 0.0
    g_use = g_avg if damping else 0.0

    linear = 1j*b_use*kappa**3 - g_use*np.abs(kappa)
    E_half = np.exp(linear*dt/2.0)

    mask = np.ones(N, dtype=bool)
    if dealias:
        mask = np.abs(mode_number) <= N/3.0

    a_hat = np.fft.fft(a0)
    if dealias:
        a_hat[~mask] = 0.0

    def nonlinear_rhs(a_hat_local):
        if not nonlinear:
            return np.zeros_like(a_hat_local)

        a_local = np.fft.ifft(a_hat_local).real

        # Conservative product:
        # -a a_z = -(1/2) d(a^2)/d zeta.
        rhs = -0.5j*kappa*np.fft.fft(a_local*a_local)
        if dealias:
            rhs[~mask] = 0.0
        return rhs

    n_steps = int(round(s_end/dt))
    snapshots = []
    initial_hat = a_hat.copy()

    for step in range(n_steps + 1):
        if step % save_every == 0 or step == n_steps:
            s = step*dt
            I2, Elow, Ehigh, Rspec, Dsum = diagnostics_from_hat(a_hat, kappa)
            gamma_exact = (
                -2.0*g_use*Dsum/I2
                if I2 > 0 and damping else 0.0
            )
            snapshots.append({
                "s": s,
                "I2": I2,
                "E_low": Elow,
                "E_high": Ehigh,
                "R_spec": Rspec,
                "Gamma_E_exact": gamma_exact,
                "a_hat": a_hat.copy(),
            })

        if step == n_steps:
            break

        # Exact half-step for the linear operator.
        a_hat = E_half*a_hat
        if dealias:
            a_hat[~mask] = 0.0

        # RK4 step for the nonlinear spectral convolution.
        k1 = nonlinear_rhs(a_hat)
        k2 = nonlinear_rhs(a_hat + 0.5*dt*k1)
        k3 = nonlinear_rhs(a_hat + 0.5*dt*k2)
        k4 = nonlinear_rhs(a_hat + dt*k3)
        a_hat = a_hat + dt*(k1 + 2*k2 + 2*k3 + k4)/6.0

        if dealias:
            a_hat[~mask] = 0.0

        # Second exact linear half-step.
        a_hat = E_half*a_hat

    return {
        "zeta": zeta,
        "kappa": kappa,
        "initial_hat": initial_hat,
        "final_hat": a_hat,
        "initial_field": a0,
        "final_field": np.fft.ifft(a_hat).real,
        "history": snapshots,
    }

print("Chapter 5 split Fourier solver ready.")

# Canonical Case B verification

Chapter 8 defines the canonical benchmark by

$$
L_g=4\pi,
\qquad
\alpha=10,
$$

$$
b_{\mathrm{avg}}=0.01,
\qquad
g_{\mathrm{avg}}=0.00505,
$$

and

$$
\widetilde a(\zeta,0)
=
\sin(0.5\zeta)
+
0.3\sin(\zeta)
+
0.1\sin(1.5\zeta).
$$

The reported converged values at $s=10$ are

$$
I_2(10)=4.984669,
$$

$$
\mathcal R_{\mathrm{spec}}(10)=10.11474
$$

for $N_\zeta=512$, $\Delta s=10^{-3}$.

The following cells reproduce the full refinement sequence before any exploratory analysis is performed.

In [ ]:
# Reproduce the Chapter 8 refinement table exactly.
caseB_runs = [
    (128, 4e-3),
    (256, 2e-3),
    (512, 1e-3),
]

verification_rows = []
solutions = {}

for N, dt in caseB_runs:
    print(f"Running N={N}, dt={dt:g}")
    result = run_ch5(
        N=N, dt=dt, s_end=10.0,
        b_avg=b_caseB, g_avg=g_caseB,
        nonlinear=True, dispersion=True, damping=True,
        dealias=True, save_every=max(1, int(round(0.1/dt))),
    )
    solutions[N] = result
    final = result["history"][-1]

    verification_rows.append({
        "N_zeta": N,
        "Delta_s": dt,
        "I2_10": final["I2"],
        "R_spec_10": final["R_spec"],
    })

verification_df = pd.DataFrame(verification_rows)
verification_df.to_csv(DATA_DIR / "ch05_caseB_refinement.csv", index=False)
display(verification_df)

print("Book reference at N=512:")
print("I2(10) = 4.984669")
print("R_spec(10) = 10.11474")

In [ ]:
# Explicit de-aliasing sensitivity check at the Chapter 8 N=256 setting.
dealiased_256 = solutions[256]["history"][-1]["R_spec"]

no_dealias_256 = run_ch5(
    N=256, dt=2e-3, s_end=10.0,
    b_avg=b_caseB, g_avg=g_caseB,
    nonlinear=True, dispersion=True, damping=True,
    dealias=False, save_every=5000,
)["history"][-1]["R_spec"]

aliasing_check = pd.DataFrame([{
    "R_spec_dealiased": dealiased_256,
    "R_spec_without_dealiasing": no_dealias_256,
    "absolute_difference": abs(dealiased_256-no_dealias_256),
}])
aliasing_check.to_csv(DATA_DIR / "ch05_aliasing_check.csv", index=False)
display(aliasing_check)

The refinement and aliasing checks are numerical verification, not physiological calibration. Their purpose is to establish that the implemented Fourier operator, nonlinear convolution, cutoff, and energy normalization reproduce the book's canonical calculation.

In [ ]:
# Reproduce the central Chapter 5 result: redistribution without growth.
caseB = solutions[512]
history = pd.DataFrame([
    {k: v for k, v in item.items() if k != "a_hat"}
    for item in caseB["history"]
])
history.to_csv(DATA_DIR / "ch05_caseB_history.csv", index=False)

# Modal contributions to I2 at s=0 and s=10.
kappa = caseB["kappa"]
coeff0 = caseB["initial_hat"]/512
coeff10 = caseB["final_hat"]/512
modal0 = L_g*np.abs(coeff0)**2
modal10 = L_g*np.abs(coeff10)**2

positive = (kappa > 0) & (kappa <= 8.0)

fig, axes = plt.subplots(1, 2, figsize=(7.25, 3.15))

axes[0].stem(
    kappa[positive], modal0[positive],
    linefmt="0.70", markerfmt="o", basefmt=" ",
    label=r"$s=0$"
)
axes[0].stem(
    kappa[positive], modal10[positive],
    linefmt="k-", markerfmt="ko", basefmt=" ",
    label=r"$s=10$"
)
axes[0].axvline(kappa_c, color=LIGHT, linestyle=":", linewidth=1.0)
axes[0].set_xlabel(r"Reduced axial wavenumber, $\kappa$")
axes[0].set_ylabel(r"Modal contribution to $I_2$")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].plot(
    history["s"], history["I2"],
    color=BLACK, linestyle="-", label=r"$I_2$"
)
ax2 = axes[1].twinx()
ax2.plot(
    history["s"], history["R_spec"],
    color=DARK, linestyle="--", label=r"$\mathcal R_{\mathrm{spec}}$"
)
axes[1].set_xlabel(r"Dimensionless time, $s$")
axes[1].set_ylabel(r"$I_2$")
ax2.set_ylabel(r"$\mathcal R_{\mathrm{spec}}$")
axes[1].spines["top"].set_visible(False)
ax2.spines["top"].set_visible(False)

# Combined legend without color dependence.
lines = axes[1].get_lines() + ax2.get_lines()
labels = [line.get_label() for line in lines]
axes[1].legend(lines, labels, frameon=False, loc="center right")

fig.tight_layout(w_pad=1.4)
save_figure(fig, "ch05_redistribution_and_decay")
plt.show()

The left panel shows the redistribution directly: initially occupied low wavenumbers seed a much broader spectrum by $s=10$.

The right panel separates two diagnostics:

- $I_2$ decreases;
- $\mathcal R_{\mathrm{spec}}$ increases.

This is the central Chapter 5 result. Spectral broadening is not the same thing as growth of the total perturbation energy.

In [ ]:
# Verify the differential energy identity through Gamma_E.
# The exact spectral formula is recorded during integration.
gamma_numeric = np.gradient(np.log(history["I2"]), history["s"])

fig, ax = plt.subplots(figsize=(5.9, 3.25))
ax.plot(
    history["s"], history["Gamma_E_exact"],
    color=BLACK, linestyle="-",
    label=r"energy identity"
)
ax.plot(
    history["s"], gamma_numeric,
    color=DARK, linestyle="--",
    label=r"finite-difference $d(\ln I_2)/ds$"
)
ax.axhline(0.0, color=LIGHT, linewidth=0.8)
ax.set_xlabel(r"Dimensionless time, $s$")
ax.set_ylabel(r"$\Gamma_E$")
ax.legend(frameon=False)
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch05_energy_identity")
plt.show()

The two evaluations of $\Gamma_E$ agree within time-discretization error and remain non-positive for the canonical positive-damping case.

This is stronger than merely observing a decreasing curve: it checks the numerical evolution against the analytical energy identity.

# Counterfactual limits

Chapter 5 separates four layers:

1. the Womersley base state;
2. the existence of an axial perturbation;
3. nonlinear mode coupling;
4. geometry-parameterized dispersion and damping.

The reduced equation permits direct mechanism-off tests:

- suppress nonlinearity: no convolution-driven transfer to new modes;
- suppress damping: conservative nonlinear–dispersive redistribution;
- suppress dispersion: nonlinear steepening plus damping;
- suppress both $b$ and $g$: inviscid Burgers-type steepening.

These are model counterfactuals, not alternate arterial models.

In [ ]:
# Mechanism-off comparisons.
counterfactual_specs = [
    ("full", True, True, True),
    ("nonlinearity off", False, True, True),
    ("damping off", True, True, False),
    ("dispersion off", True, False, True),
]

counter_rows = []
counter_results = {}

for label, nl_on, disp_on, damp_on in counterfactual_specs:
    print("Running:", label)
    result = run_ch5(
        N=256, dt=2e-3, s_end=10.0,
        b_avg=b_caseB, g_avg=g_caseB,
        nonlinear=nl_on,
        dispersion=disp_on,
        damping=damp_on,
        dealias=True,
        save_every=50,
    )
    counter_results[label] = result
    final = result["history"][-1]
    counter_rows.append({
        "case": label,
        "I2_10": final["I2"],
        "R_spec_10": final["R_spec"],
    })

counter_df = pd.DataFrame(counter_rows)
counter_df.to_csv(DATA_DIR / "ch05_counterfactuals.csv", index=False)
display(counter_df)

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.1))

styles = [
    (BLACK, "-"),
    (DARK, "--"),
    (MID, "-."),
    (LIGHT, ":"),
]

for (label, result), (gray, ls) in zip(counter_results.items(), styles):
    hist = pd.DataFrame([
        {k: v for k, v in item.items() if k != "a_hat"}
        for item in result["history"]
    ])
    axes[0].plot(hist["s"], hist["I2"], color=gray, linestyle=ls, label=label)
    axes[1].plot(hist["s"], hist["R_spec"], color=gray, linestyle=ls, label=label)

axes[0].set_xlabel(r"Dimensionless time, $s$")
axes[0].set_ylabel(r"$I_2$")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].set_xlabel(r"Dimensionless time, $s$")
axes[1].set_ylabel(r"$\mathcal R_{\mathrm{spec}}$")
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch05_counterfactual_mechanisms")
plt.show()

The most important mechanism-off result is the nonlinearity-off case: without the quadratic convolution, the initially empty high-wavenumber band is not populated by mode coupling.

The damping-off case separates redistribution from global decay. The dispersive term affects phase organization but does no direct work on $I_2$ in the constant-coefficient periodic system.

# Womersley-dependent reduced coefficients in the VascuQuest population

The book defines

$$
b_0=b_{\mathrm{ref}}\alpha^{-2},
$$

and

$$
g_0=g_{\mathrm{ref}}
\left(
1+\frac{C_g}{\alpha}
\right).
$$

VascuQuest can therefore supply a physiological distribution of $\alpha$ that is passed through these **already defined book relations**.

This operation is valid without inventing a geometry closure because it uses only the explicit $\alpha$ dependence of the canonical reduced coefficients.

In [ ]:
# Build the PWDB subject/site alpha population.
meta = subject_meta.set_index("subject_id")
population_rows = []

for site in SITES:
    print("Processing", SITE_LABELS[site])
    ids_a, A_matrix = load_waveform_matrix(site, "A")

    for sid, A_row in zip(ids_a, A_matrix):
        if sid not in meta.index:
            continue

        valid = np.isfinite(A_row)
        if valid.sum() < 16:
            continue

        R = np.sqrt(np.mean(A_row[valid])/np.pi)
        heart_rate = float(meta.loc[sid, "heart_rate_bpm"])
        age = float(meta.loc[sid, "age_years"])

        T = 60.0/heart_rate
        Omega = 2.0*np.pi/T
        alpha = R*np.sqrt(Omega/nu)

        b0 = b_ref*alpha**-2
        g0 = g_ref*(1.0 + C_g/alpha)

        population_rows.append({
            "subject_id": sid,
            "age_years": age,
            "site": site,
            "R_m": R,
            "heart_rate_bpm": heart_rate,
            "alpha": alpha,
            "b0": b0,
            "g0": g0,
        })

population_df = pd.DataFrame(population_rows)
population_df.to_csv(DATA_DIR / "ch05_population_reduced_coefficients.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.1))

for site, gray, marker in zip(
    SITES,
    ["0.0","0.18","0.30","0.42","0.54","0.66","0.78"],
    ["o","s","^","D","v","P","X"],
):
    sub = population_df.loc[population_df["site"] == site]
    axes[0].scatter(
        sub["alpha"], sub["b0"],
        s=8, marker=marker, facecolors="none", edgecolors=gray,
        linewidths=0.45, label=SITE_LABELS[site]
    )
    axes[1].scatter(
        sub["alpha"], sub["g0"],
        s=8, marker=marker, facecolors="none", edgecolors=gray,
        linewidths=0.45, label=SITE_LABELS[site]
    )

axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel(r"Womersley number, $\alpha$")
axes[0].set_ylabel(r"$b_0=b_{\mathrm{ref}}\alpha^{-2}$")
clean_axes(axes[0])

axes[1].set_xscale("log")
axes[1].set_xlabel(r"Womersley number, $\alpha$")
axes[1].set_ylabel(r"$g_0=g_{\mathrm{ref}}(1+C_g/\alpha)$")
axes[1].legend(frameon=False, ncol=2)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch05_vascuquest_coefficient_population")
plt.show()

This figure is a direct mapping of VascuQuest-derived $\alpha$ through the Chapter 5 coefficient formulas.

It does not use PWDB taper, curvature, or any other geometry descriptor to calibrate $b$ or $g$. The geometry-to-coefficient closure remains deliberately separate.

# How the canonical spectral response changes with $\alpha$

Because the canonical coefficients depend on $\alpha$, the reduced model predicts different balances of dispersion and damping as $\alpha$ changes.

To expose that dependence without running thousands of population simulations, the notebook computes a deterministic response curve over the VascuQuest-supported $\alpha$ range, using the same initial condition, domain, cutoff, and numerical method as Case B.

The response curve is then used only for visualization of where the PWDB population lies relative to the model response.

In [ ]:
# Deterministic alpha-response sweep.
alpha_min = max(1.0, 0.9*population_df["alpha"].min())
alpha_max = 1.1*population_df["alpha"].max()
alpha_grid = np.geomspace(alpha_min, alpha_max, 24)

alpha_response_rows = []

for j, alpha in enumerate(alpha_grid):
    b0 = b_ref*alpha**-2
    g0 = g_ref*(1.0 + C_g/alpha)

    print(f"alpha sweep {j+1}/{len(alpha_grid)}: {alpha:.3f}")
    result = run_ch5(
        N=256, dt=2e-3, s_end=10.0,
        b_avg=b0, g_avg=g0,
        nonlinear=True, dispersion=True, damping=True,
        dealias=True, save_every=5000,
    )
    final = result["history"][-1]

    alpha_response_rows.append({
        "alpha": alpha,
        "b0": b0,
        "g0": g0,
        "I2_10": final["I2"],
        "R_spec_10": final["R_spec"],
        "Gamma_E_10": final["Gamma_E_exact"],
    })

alpha_response_df = pd.DataFrame(alpha_response_rows)
alpha_response_df.to_csv(DATA_DIR / "ch05_alpha_response.csv", index=False)

# Monotone shape-preserving interpolation is used only to place population
# alpha values onto the computed response curve for descriptive comparison.
interp_I2 = PchipInterpolator(alpha_response_df["alpha"], alpha_response_df["I2_10"])
interp_R = PchipInterpolator(alpha_response_df["alpha"], alpha_response_df["R_spec_10"])

population_df["I2_10_model"] = interp_I2(population_df["alpha"])
population_df["R_spec_10_model"] = interp_R(population_df["alpha"])

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.15))

axes[0].plot(
    alpha_response_df["alpha"], alpha_response_df["I2_10"],
    color=BLACK
)
axes[0].scatter(
    population_df["alpha"], population_df["I2_10_model"],
    s=6, facecolors="none", edgecolors=LIGHT, linewidths=0.4
)
axes[0].set_xscale("log")
axes[0].set_xlabel(r"Womersley number, $\alpha$")
axes[0].set_ylabel(r"Model $I_2(10)$")
clean_axes(axes[0])

axes[1].plot(
    alpha_response_df["alpha"], alpha_response_df["R_spec_10"],
    color=BLACK
)
axes[1].scatter(
    population_df["alpha"], population_df["R_spec_10_model"],
    s=6, facecolors="none", edgecolors=LIGHT, linewidths=0.4
)
axes[1].set_xscale("log")
axes[1].set_xlabel(r"Womersley number, $\alpha$")
axes[1].set_ylabel(r"Model $\mathcal R_{\mathrm{spec}}(10)$")
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch05_vascuquest_alpha_response_context")
plt.show()

The black curves are reduced-model calculations. The open points are the VascuQuest-derived $\alpha$ values projected onto those curves.

The correct interpretation is therefore:

> if the canonical Chapter 5 coefficient formulas and initial condition are retained, VascuQuest-derived pulsatile scales place different subject/site pairs at different locations on the reduced-model response curve.

This is not a claim that the virtual subjects actually exhibit the plotted $I_2$ or $\mathcal R_{\mathrm{spec}}$ values.

In [ ]:
# Age-group comparison of the same model projection.
AGE_SITES = ["AorticRoot", "Carotid", "Femoral", "Radial"]
age_summary = (
    population_df.loc[population_df["site"].isin(AGE_SITES)]
    .groupby(["age_years", "site"])
    .agg(
        alpha_median=("alpha", "median"),
        R_spec_10_median=("R_spec_10_model", "median"),
    )
    .reset_index()
)

styles_age = [
    (BLACK, "-", "o"),
    (DARK, "--", "s"),
    (MID, "-.", "^"),
    (LIGHT, ":", "D"),
]

fig, ax = plt.subplots(figsize=(6.0, 3.35))

for site, (gray, ls, marker) in zip(AGE_SITES, styles_age):
    sub = age_summary.loc[age_summary["site"] == site]
    ax.plot(
        sub["age_years"], sub["R_spec_10_median"],
        color=gray, linestyle=ls, marker=marker, markersize=4,
        label=SITE_LABELS[site]
    )

ax.set_xlabel("PWDB source age (years)")
ax.set_ylabel(r"Median model $\mathcal R_{\mathrm{spec}}(10)$")
ax.legend(frameon=False)
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch05_age_group_model_response")
plt.show()

The age-group comparison is descriptive and doubly qualified:

1. age is only a PWDB grouping variable that shifts the population distribution of $R$, heart rate, and therefore $\alpha$;
2. $\mathcal R_{\mathrm{spec}}(10)$ is the output of the canonical reduced model projected from that $\alpha$.

It is not an observed age-dependent spectral-broadening law.

# Spatial averaging is not a homogenization theorem

The conceptual model permits spatial modulation through

$$
b(\zeta)=b_0+\widetilde b(\zeta),
\qquad
g(\zeta)=g_0+\widetilde g(\zeta),
$$

with zero-mean fluctuations.

The implemented solver advances only

$$
b_{\mathrm{avg}}=b_0,
\qquad
g_{\mathrm{avg}}=g_0.
$$

Therefore two different zero-mean modulation patterns with the same spatial means produce the **same canonical implemented evolution**.

This is a statement about the current implementation. It is not evidence that spatially varying coefficients are dynamically equivalent in the unreduced variable-coefficient equation.

In [ ]:
# Visualize the distinction between conceptual variable coefficients
# and the coefficients actually supplied to the canonical solver.
#
# The modulation values below are deliberately illustrative only. Their
# spatial means are exactly the Case B coefficients.
zeta = np.linspace(0.0, L_g, 800)
eps_b_demo = 0.35
eps_g_demo = 0.25
kappa_g_demo = 1.0

b_variable = b_caseB*(1.0 + eps_b_demo*np.cos(kappa_g_demo*zeta))
g_variable = g_caseB*(1.0 + eps_g_demo*np.cos(kappa_g_demo*zeta))

fig, axes = plt.subplots(1, 2, figsize=(7.15, 3.0))

axes[0].plot(zeta, b_variable, color=BLACK, label=r"conceptual $b(\zeta)$")
axes[0].axhline(b_caseB, color=DARK, linestyle="--",
                label=r"implemented $b_{\mathrm{avg}}$")
axes[0].set_xlabel(r"Dimensionless axial coordinate, $\zeta$")
axes[0].set_ylabel(r"Reduced dispersive coefficient")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].plot(zeta, g_variable, color=BLACK, label=r"conceptual $g(\zeta)$")
axes[1].axhline(g_caseB, color=DARK, linestyle="--",
                label=r"implemented $g_{\mathrm{avg}}$")
axes[1].set_xlabel(r"Dimensionless axial coordinate, $\zeta$")
axes[1].set_ylabel(r"Reduced damping coefficient")
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch05_conceptual_vs_implemented_coefficients")
plt.show()

The modulation in this figure is illustrative only and is not used in the time integration.

Its purpose is to prevent a common interpretive error: the geometry-parameterized derivation and the spatially averaged canonical implementation are distinct modeling levels.

# What the reduced model establishes

The notebook reproduces the Chapter 5 controlled claim:

- quadratic steepening transfers content among axial Fourier wavenumbers;
- dispersion changes phase without directly removing $I_2$ in the constant-coefficient periodic model;
- positive fractional damping enforces non-increasing $I_2$;
- therefore $\mathcal R_{\mathrm{spec}}$ can rise strongly while $\Gamma_E<0$ and $I_2$ decreases.

The reduced model does not resolve Dean vortices, separation, local three-dimensional curvature dynamics, patient-specific secondary flow, wall motion, or broadband transition physics.

The phrase **geometry-parameterized** therefore refers to the prescribed coefficient architecture, not to direct simulation of resolved anatomy.

# What the reader should learn

1. **Chapter 5 restores an axial degree of freedom that Womersley flow suppresses.** The perturbation amplitude $\widetilde a(\zeta,s)$ can evolve along the vessel.

2. **Spatial wavenumber $\kappa$ is not temporal harmonic number $m$.** Chapter 5 studies redistribution across axial Fourier modes.

3. **The nonlinear term is the mode mixer.** Its Fourier transform is a convolution.

4. **Dispersion and damping have different spectral signatures.** The former is imaginary in the linear Fourier symbol; the latter is real and non-positive.

5. **Spectral broadening and total-energy growth are different questions.** $\mathcal R_{\mathrm{spec}}$ can increase while $I_2$ decreases.

6. **The analytical energy identity constrains interpretation.** For $g_{\mathrm{avg}}\ge0$, the implemented periodic system cannot exhibit positive total quadratic-energy growth.

7. **The canonical implementation advances spatially averaged coefficients.** It does not resolve the variable coefficient field in time stepping and should not be described as a proven homogenized model.

8. **VascuQuest geometry motivates the need for a geometry-to-coefficient map but does not supply that map.**

9. **VascuQuest-derived $\alpha$ can legitimately enter the book-defined $b_0(\alpha)$ and $g_0(\alpha)$ relations.** The resulting spectral diagnostics are model projections, not observed arterial spectra.

# Chapter-enrichment candidates

The notebook produces eight principal figures.

**Candidate 1 — VascuQuest geometry scales.**  
Already introduced into the book package as a geometry-context figure. Retain only if it materially improves the explanation of why $\mathcal M_G$ is required.

**Candidate 2 — redistribution and decay.**  
This reproduces the chapter's existing Case B result. Use as a replacement only, not as an additional figure.

**Candidate 3 — energy-identity verification.**  
Strong notebook verification; probably unnecessary in the book unless the distinction between numerical observation and analytical decay constraint needs stronger visual emphasis.

**Candidate 4 — mechanism-off counterfactuals.**  
Strong book candidate. It cleanly separates nonlinear transfer, damping, and dispersion.

**Candidate 5 — VascuQuest coefficient population.**  
Potential book candidate if the chapter benefits from showing the physiological $\alpha$ range entering the explicit $b_0(\alpha)$ and $g_0(\alpha)$ parameterization.

**Candidate 6 — VascuQuest $\alpha$ response context.**  
Potentially strong, but only if its model-projection status is stated prominently. It should not be presented as a measured population spectral response.

**Candidate 7 — age-group model response.**  
Notebook-only unless a clear, nonredundant age-stratified pattern appears after execution.

**Candidate 8 — conceptual versus implemented coefficients.**  
Strong editorial/scientific candidate because it makes the chapter's most important implementation qualification visually explicit: variable-coefficient motivation versus spatially averaged time stepping.

The chapter's existing spectral-mechanism schematic is supported by these calculations and does not require a duplicate numerical figure.

No figure is promoted automatically. Book insertion requires a nonredundant scientific gain.

In [ ]:
# Reproducibility record.
manifest = {
    "book": "Nonlinear Arterial Hemodynamics",
    "chapter": 5,
    "chapter_title": "Geometry-Parameterized Spectral Dynamics",
    "vascuquest_git_ref": VQ_GIT_REF,
    "pwdb_record_id": PWDB_RECORD_ID,
    "pwdb_doi": PWDB_DOI,
    "rho_kg_m3": rho,
    "mu_Pa_s": mu,
    "nu_m2_s": nu,
    "L_g": L_g,
    "alpha_caseB": alpha_caseB,
    "b_ref": b_ref,
    "g_ref": g_ref,
    "C_g": C_g,
    "b_caseB": b_caseB,
    "g_caseB": g_caseB,
    "kappa_c": kappa_c,
    "representative_subject_id": representative_subject,
    "representative_age_years": float(target_age),
    "sites": SITES,
    "source_age_strata_years": [float(x) for x in source_ages],
    "caseB_N512_I2_10": float(verification_df.loc[
        verification_df["N_zeta"] == 512, "I2_10"
    ].iloc[0]),
    "caseB_N512_R_spec_10": float(verification_df.loc[
        verification_df["N_zeta"] == 512, "R_spec_10"
    ].iloc[0]),
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "execution_status": "completed to this cell",
    "qualification": (
        "PWDB geometry is used only to document arterial geometric scales. "
        "No geometry-to-coefficient calibration is inferred. PWDB-derived alpha "
        "is passed through the explicit Chapter 5 canonical b0(alpha) and g0(alpha) "
        "relations; resulting I2 and R_spec values are reduced-model projections, "
        "not observed arterial spectral diagnostics."
    ),
}
(META_DIR / "reproducibility_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print(json.dumps(manifest, indent=2))
print("\nGenerated PDF figures:")
for path in sorted(FIG_DIR.glob("*.pdf")):
    print(" -", path.name)

# Reproducibility record

A successful **Run all** execution writes:

- B&W vector PDF figures and high-resolution PNG previews;
- VascuQuest source-geometry descriptors;
- the complete Case B refinement and aliasing checks;
- canonical time-history diagnostics;
- mechanism-off counterfactual results;
- PWDB-derived $\alpha$, $b_0$, and $g_0$ population tables;
- the deterministic $\alpha$-response sweep;
- VascuQuest/PWDB verification metadata;
- a final reproducibility manifest.

The notebook contains no hidden geometry calibration, interactive branch, manually selected subject, or undocumented alteration of the Chapter 5 solver.